# Daily Challenge – Fine-tuning avec LoRA (PEFT)

**Dataset :** `Abirate/english_quotes` — 10% du split training  
**Modèle :** `bigscience/bloomz-560m`  
**Objectif :** Fine-tuner un LLM avec LoRA (Low-Rank Adaptation) via PEFT pour générer des citations en anglais.

## Étape 1 – Installation des librairies

In [ ]:
%pip install peft==0.4.0 --quiet
%pip install datasets --quiet

## Étape 2 – Création du dossier cache

In [ ]:
import os
os.makedirs('../cache', exist_ok=True)
print('Dossier cache créé.')

## Étape 3 – Chargement du modèle et du tokenizer

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'bigscience/bloomz-560m'

tokenizer        = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(f'Modèle chargé : {model_name}')
print(f'Paramètres totaux : {sum(p.numel() for p in foundation_model.parameters()):,}')

## Étape 4 – Chargement et prétraitement du dataset

In [ ]:
# Chargement de 10% du split training
data = load_dataset('Abirate/english_quotes', split='train[:10%]')
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)

train_sample = data.select(range(5))
display(train_sample.to_pandas()[['quote']])

print(f'\nNombre total d\'exemples (10%) : {len(data)}')

## Étape 5 – Configuration LoRA

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

# Inspection des noms de couches pour identifier les modules cibles
print('Modules linéaires disponibles :')
for name, module in foundation_model.named_modules():
    if isinstance(module, __import__('torch').nn.Linear):
        print(' ', name)

In [ ]:
lora_config = LoraConfig(
    r=1,                          # rang faible : peu de paramètres entraînables
    lora_alpha=1,                 # facteur d'échelle (souvent = r)
    target_modules=['query_key_value'],  # couches attention de BLOOM
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

# Application des adaptateurs LoRA au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

## Étape 6 – Arguments d'entraînement + Trainer

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join('../cache/working', 'peft_lab_outputs')

training_args = TrainingArguments(
    report_to='none',
    output_dir=output_directory,
    auto_find_batch_size=True,    # trouve automatiquement le batch size optimal
    learning_rate=3e-2,           # LR plus élevé que le fine-tuning complet (LoRA converge vite)
    num_train_epochs=1,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()
print('Entraînement terminé.')

## Étape 7 – Sauvegarde du modèle LoRA

In [ ]:
import time

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f'peft_model_{time_now}')
trainer.model.save_pretrained(peft_model_path)

print(f'Modèle LoRA sauvegardé dans : {peft_model_path}')
print('Fichiers :', os.listdir(peft_model_path))

## Étape 8 – Chargement pour inférence et génération de texte

In [ ]:
from peft import PeftModel

# Rechargement du modèle de base + adaptateurs LoRA (is_trainable=False)
base_model  = AutoModelForCausalLM.from_pretrained(model_name)
loaded_peft = PeftModel.from_pretrained(base_model, peft_model_path, is_trainable=False)
loaded_peft.eval()

print('Modèle LoRA chargé pour inférence.')

In [ ]:
# Génération de texte avec le modèle fine-tuné
prompts = [
    'Two things are infinite: ',
    'The secret of life is ',
    'In the middle of difficulty ',
]

for prompt in prompts:
    inputs  = tokenizer(prompt, return_tensors='pt')
    outputs = loaded_peft.generate(
        input_ids=inputs['input_ids'],
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
    )
    generated = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    print(f'Prompt   : {prompt}')
    print(f'Génération : {generated}')
    print()

## Réflexion – Pourquoi LoRA ?

### Principe de LoRA
Au lieu de mettre à jour tous les poids W (in_dim × out_dim), LoRA décompose la mise à jour en deux matrices de faible rang : **W + ΔW = W + A × B** où A est (in_dim × r) et B est (r × out_dim) avec r ≪ min(in_dim, out_dim).

### Avantages observés

| Aspect | Fine-tuning complet | LoRA (r=1) |
|---|---|---|
| **Paramètres entraînables** | ~560M | < 1M (< 0.2%) |
| **Mémoire GPU nécessaire** | ~4GB+ | < 500MB |
| **Vitesse d'entraînement** | Lente | 5-10× plus rapide |
| **Performances** | Référence | Proche du fine-tuning complet |
| **Stockage** | Copie complète du modèle | Seulement les matrices A et B |

### Interprétation
- Avec r=1 et `num_train_epochs=1`, le modèle fine-tuné génère des textes qui ressemblent davantage au style des citations du dataset (plus aphoristiques, plus formels).
- Augmenter r (ex. r=4 ou r=8) permettrait de capturer des adaptations plus fines mais au coût de plus de paramètres.
- `target_modules=['query_key_value']` cible les couches d'attention de BLOOM : c'est là que réside la majorité de la capacité de modélisation du langage.